# 1. CSV 전용 테스트 노트북

**식샤 API·식샤 원본 코드 미사용.** 직접 수집한 학식/위치/운영시간을 **CSV·JSON 파일로만** 저장합니다. DB 접근 코드를 호출하지 않습니다.

- `MODE='demo'`: 합성 HTML로 파싱/저장을 안전하게 확인합니다.
- `MODE='live'`: 실제 서울대 웹사이트에 요청합니다. 실제 재사용/수집 전 사이트 이용 조건을 확인하세요.
- live에서는 KST 오늘부터 최대 30일간 요청합니다. 아직 게시되지 않은 식단은 생성하지 않습니다.

In [1]:
from pathlib import Path
import sys, json, csv
from datetime import date, datetime
from zoneinfo import ZoneInfo
HERE=Path.cwd().resolve()
ROOT=next((d for d in (HERE, HERE.parent, HERE.parent.parent) if (d/'src/chajwo_meals').is_dir()),None)
if ROOT is None:
    raise RuntimeError('프로젝트 루트 또는 notebooks 폴더에서 실행하세요.')
sys.path.insert(0,str(ROOT/'src'))
print('ROOT:', ROOT)


ROOT: C:\Users\rhrnf\크롤링


## 2. 테스트 조건

In [12]:
MODE='live'  # 실제 수집을 원할 때 'live'로 명시적 변경
DAYS=30     # 하루치로 검증한 후 DAYS=30으로 변경
SOURCES=('snuco','snudorm','vet')
START_DATE=date(2026,10,22) if MODE=='demo' else datetime.now(ZoneInfo('Asia/Seoul')).date()
OUTPUT=ROOT/'outputs'/('csv_demo' if MODE=='demo' else 'csv_live')
print('MODE:',MODE,'START_DATE:',START_DATE,'DAYS:',DAYS,'OUTPUT:',OUTPUT)


MODE: live START_DATE: 2026-10-22 DAYS: 30 OUTPUT: C:\Users\rhrnf\크롤링\outputs\csv_live


## 3. 테스트 HTML (demo 모드 전용)

In [13]:
from urllib.parse import urlsplit, parse_qs
class FixtureFetcher:
    """실제 네트워크 대신 자체 작성한 테스트 HTML/JSON을 반환합니다."""
    def __init__(self, root):
        self.root=root
        self.requests=[]
    def get(self, url):
        self.requests.append(url)
        if 'snuco.snu.ac.kr/foodmenu/' in url:
            name='snuco.html'
        elif 'snudorm.snu.ac.kr/foodmenu/' in url:
            name='snudorm.html'
        elif 'vet.snu.ac.kr/cafe_menu/' in url:
            name='vet.html'
        elif '/식당안내/' in url:
            name='guide_demo.html'
        elif 'map.snu.ac.kr/api/building.action' in url:
            name='campus_map_demo.json'
        else:
            raise ValueError(f'예상하지 않은 요청: {url}')
        html=(self.root/name).read_text(encoding='utf-8')
        if name in ('snuco.html','snudorm.html'):
            day=parse_qs(urlsplit(url).query)['date'][0]
            html=html.replace('2026-10-22', day)
        return html


## 4. 식단·식당 정보 수집

In [14]:
from chajwo_meals.pipeline import collect, collect_metadata
from chajwo_meals.http import RateLimitedFetcher
fetcher=FixtureFetcher(ROOT/'tests'/'fixtures') if MODE=='demo' else RateLimitedFetcher()
rows, errors = collect(start=START_DATE, days=DAYS, sources=SOURCES, fetcher=fetcher)
restaurant_directory, metadata_errors=collect_metadata(fetcher=fetcher)
errors.extend(metadata_errors)
print('원문 메뉴 줄 수:',len(rows),'식당/코너 수:',len(restaurant_directory),'오류 수:',len(errors))
if errors: print('오류 예시:',errors[:3])


원문 메뉴 줄 수: 8264 식당/코너 수: 33 오류 수: 26
오류 예시: [{'source': 'snuco', 'date': '2026-11-14', 'error_type': 'PageStructureError', 'message': 'snuco: no restaurant rows'}, {'source': 'snuco', 'date': '2026-11-15', 'error_type': 'PageStructureError', 'message': 'snuco: no restaurant rows'}, {'source': 'snuco', 'date': '2026-11-16', 'error_type': 'PageStructureError', 'message': 'snuco: no restaurant rows'}]


## 5. CSV·JSON 파일만 저장

In [15]:
from chajwo_meals.pipeline import export
report=export(rows,errors,OUTPUT,started=START_DATE,days=DAYS,sources=SOURCES,restaurant_directory=restaurant_directory)
report['storage']='csv'
(OUTPUT/'manifest.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('저장 파일:', sorted(p.name for p in OUTPUT.iterdir() if p.is_file()))
print('manifest:',json.dumps(report,ensure_ascii=False,indent=2)[:1200])


저장 파일: ['errors.json', 'fixed_menu_catalog.csv', 'fixed_menu_observed.csv', 'locations.csv', 'manifest.json', 'meal_service_hours.csv', 'menu_entries.csv', 'menu_items.csv', 'menu_lines.csv', 'raw_records.json', 'restaurant_directory.csv', 'siksha_seed_comparison.csv']
manifest: {
  "project": "Chajwo independent crawler v0.3.2",
  "generated_at_utc": "2026-10-22T10:35:24.857603+00:00",
  "start": "2026-10-22",
  "days": 30,
  "end": "2026-11-20",
  "sources": [
    "snuco",
    "snudorm",
    "vet"
  ],
  "observation_count": 8264,
  "entry_count": 2013,
  "priced_item_count": 1748,
  "dated_item_count": 1614,
  "fixed_catalog_rows": 136,
  "fixed_catalog_needs_review": 2,
  "fixed_page_observation_lines": 5856,
  "restaurant_count": 33,
  "guide_matched_units": 31,
  "coordinates_matched_units": 32,
  "price_autocorrections": 34,
  "meal_hour_notice_count": 417,
  "errors": 26,
  "by_source": {
    "snuco": 8116,
    "snudorm": 146,
    "vet": 2
  },
  "unknown_locations": [
    "기숙사

## 6. CSV 내용 확인

In [16]:
for filename in ('menu_items.csv','restaurant_directory.csv','meal_service_hours.csv'):
    print('\n파일:', filename)
    with (OUTPUT/filename).open(newline='',encoding='utf-8-sig') as f:
        reader=csv.DictReader(f)
        print('컬럼:',reader.fieldnames)
        for row in list(reader)[:3]:
            print(row)
assert (OUTPUT/'menu_items.csv').exists()
assert (OUTPUT/'restaurant_directory.csv').exists()
assert report['restaurant_count']==33
print('CSV 전용 확인 완료: 이 노트북은 DB 파일을 열거나 수정하지 않습니다.')



파일: menu_items.csv
컬럼: ['record_id', 'source_date', 'source', 'building_number', 'building_name', 'restaurant', 'meal_type', 'menu_name', 'price_won', 'no_meat_label', 'source_url', 'price_variant', 'data_origin', 'price_status', 'restaurant_unit', 'price_raw']
{'record_id': 'c2084f0cba31505efac7', 'source_date': '2026-10-22', 'source': 'snuco', 'building_number': '63동', 'building_name': '학생회관', 'restaurant': '학생회관식당', 'meal_type': 'BREAKFAST', 'menu_name': '버섯스크램블에그', 'price_won': '3000', 'no_meat_label': '', 'source_url': 'https://snuco.snu.ac.kr/foodmenu/?date=2026-10-22', 'price_variant': '', 'data_origin': 'observed_page', 'price_status': 'as_published', 'restaurant_unit': '학생회관식당', 'price_raw': '3,000원'}
{'record_id': 'fc2d4b79abf4db8a84c6', 'source_date': '2026-10-22', 'source': 'snuco', 'building_number': '63동', 'building_name': '학생회관', 'restaurant': '학생회관식당', 'meal_type': 'LUNCH', 'menu_name': '제육고추장볶음', 'price_won': '6000', 'no_meat_label': '', 'source_url': 'https://snuco.s

## 7. 라이브 데이터 품질 확인

`manifest.json`의 `errors`, `observation_count`, `guide_matched_units`, `coordinates_matched_units`를 확인하세요. CSV가 생성됐다는 사실은 HTTP 요청 성공을 보장하지 않습니다. 실패한 경우 `errors.json`을 읽으세요.